<a href="https://colab.research.google.com/github/tkoganti/tkoganti.github.io/blob/master/custom_variant_scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import zipfile
import os
from tqdm import tqdm
from alphagenome.data import genome
from alphagenome_research.model import dna_model
from alphagenome.models import variant_scorers


from google.colab import files

# Upload your variant text file
uploaded = files.upload()
variants_file = list(uploaded.keys())[0]
print(f"Uploaded: {variants_file}")

# Upload your checkpoint zip
print("\nUpload your checkpoint zip...")
uploaded = files.upload()
checkpoint_zip = list(uploaded.keys())[0]
print(f"Uploaded: {checkpoint_zip}")

/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.31.1 at alphagenome/protos/dna_model.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.31.1 at alphagenome/protos/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  wa

Saving IRF4_var_AGinput.tsv to IRF4_var_AGinput.tsv
Uploaded: IRF4_var_AGinput.tsv

Upload your checkpoint zip...


Saving checkpoint.zip to checkpoint.zip
Uploaded: checkpoint.zip


In [4]:
# Upgrade protobuf to match gencode version
!pip install protobuf==6.31.1
print("Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 29.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.31.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.31.1 which is incompatible.


Done!


In [1]:

from IPython.display import clear_output

!PIP_NO_BINARY=pyBigWig pip install \
    git+https://github.com/google-deepmind/alphagenome_research.git


print("AlphaGenome installed!")

# Verify
import alphagenome
import alphagenome_research
print("alphagenome version:", alphagenome.__version__)

  Cloning https://github.com/google-deepmind/alphagenome_research.git to /tmp/pip-req-build-qzvochsp
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/alphagenome_research.git /tmp/pip-req-build-qzvochsp
  Resolved https://github.com/google-deepmind/alphagenome_research.git to commit dad09dd11a480fb8c16ae703a606f55a6e3e968b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
AlphaGenome installed!
alphagenome version: 0.6.1


In [2]:
# Extract checkpoint model
print("\nExtracting checkpoint...")
os.makedirs('/content/checkpoint_mm', exist_ok=True)
with zipfile.ZipFile(f'/content/{checkpoint_zip}', 'r') as z:
    z.extractall('/content/checkpoint_mm/')
print("Extracted!")
!ls /content/checkpoint_mm/


Extracting checkpoint...
Extracted!
array_metadatas       d		      _METADATA        _sharding
_CHECKPOINT_METADATA  manifest.ocdbt  ocdbt.process_0


In [3]:
# Apply splice patch
filepath = '/usr/local/lib/python3.12/dist-packages/alphagenome_research/model/dna_model.py'

with open(filepath, 'r') as f:
    content = f.read()

old_code = """  reference_splice_sites = (
      reference_predictions['splice_sites_classification']['predictions']
      * splice_junction_masks.reference_genes
  )
  alternate_splice_sites = alternate_predictions['splice_sites_classification'][
      'predictions'
  ]"""

new_code = """  if 'splice_sites_classification' in reference_predictions:
    reference_splice_sites = (
        reference_predictions['splice_sites_classification']['predictions']
        * splice_junction_masks.reference_genes
    )
    alternate_splice_sites = alternate_predictions['splice_sites_classification'][
        'predictions'
    ]
  else:
        reference_splice_sites = None
    alternate_splice_sites = None"""

new_content = content.replace(old_code, new_code)
if new_content != content:
    with open(filepath, 'w') as f:
        f.write(new_content)
    print("✅ Splice patch applied!")
else:
    print("Already patched!")


✅ Splice patch applied!


In [4]:
# load variants
df_variants = pd.read_csv(f'/content/{variants_file}', sep='\t')
print(f"Loaded {len(df_variants)} variants")
print(df_variants.head())

# Use only first 4 columns
vcf = pd.DataFrame({
    'variant_id': df_variants['chrom'].astype(str) + '_' +
                  df_variants['pos'].astype(str) + '_' +
                  df_variants['ref'] + '_' +
                  df_variants['alt'],
    'CHROM': df_variants['chrom'].apply(
        lambda x: x if str(x).startswith('chr') else f'chr{x}'
    ),
    'POS': df_variants['pos'].astype(int),
    'REF': df_variants['ref'],
    'ALT': df_variants['alt'],
})
print(f"\nFormatted {len(vcf)} variants")
print(vcf.head())

Loaded 70 variants
  chrom     pos ref alt sample_name                                   gene
0  chr6  229915   T   C    BM068331  416J7.5-DUSP22,DUSP22,RP3,RP3-416J7.5
1  chr6  301351   G   A    BM072030                                 DUSP22
2  chr6  256528   T   C    BM042351  416J7.5-DUSP22,DUSP22,RP3,RP3-416J7.5
3  chr6  313674   G   T    BM062926                                 DUSP22
4  chr6  362066   C   T    BM049787                                   IRF4

Formatted 70 variants
        variant_id CHROM     POS REF ALT
0  chr6_229915_T_C  chr6  229915   T   C
1  chr6_301351_G_A  chr6  301351   G   A
2  chr6_256528_T_C  chr6  256528   T   C
3  chr6_313674_G_T  chr6  313674   G   T
4  chr6_362066_C_T  chr6  362066   C   T


In [5]:
uploaded = files.upload()
track_metadata_file = list(uploaded.keys())[0]
print(f"Uploaded: {track_metadata_file}")

# Load it
TRACK_METADATA = pd.read_csv(f'/content/{track_metadata_file}')
print("TRACK_METADATA loaded:")
print(TRACK_METADATA)

Saving track_metadata.csv to track_metadata.csv
Uploaded: track_metadata.csv
TRACK_METADATA loaded:
   output_type                                               name strand  \
0      RNA_SEQ  MM_BM061418.Aligned.sortedByCoord.out total RN...      .   
1      RNA_SEQ  MM_BM061472.Aligned.sortedByCoord.out total RN...      .   
2      RNA_SEQ  MM_BM63568R.Aligned.sortedByCoord.out total RN...      .   
3      RNA_SEQ  MM_BM64752R.Aligned.sortedByCoord.out total RN...      .   
4      RNA_SEQ  MM_BM65549R.Aligned.sortedByCoord.out total RN...      .   
5      RNA_SEQ  MM_BM65562R.Aligned.sortedByCoord.out total RN...      .   
6      RNA_SEQ  MM_BM65678R.Aligned.sortedByCoord.out total RN...      .   
7      RNA_SEQ  MM_BM65904R.Aligned.sortedByCoord.out total RN...      .   
8      RNA_SEQ  MM_BM66361R.Aligned.sortedByCoord.out total RN...      .   
9      RNA_SEQ  MM_BM67614R.Aligned.sortedByCoord.out total RN...      .   
10     RNA_SEQ  MM_BM68182R.Aligned.sortedByCoord.out total RN..

In [6]:
# Rebuild TRACK METADATA
TRACK_METADATA = pd.read_csv('/content/track_metadata.csv')

print("\nTrack metadata:")
print(TRACK_METADATA)



# Rebuild output metadata
import dataclasses
import orbax.checkpoint as ocp
from alphagenome_research.model.metadata import metadata as metadata_lib


def build_output_metadata(track_metadata):
    metadata = {}
    for output_type, df_group in track_metadata.groupby('output_type'):
        output_type_enum = dna_model.OutputType[str(output_type)]
        metadata[output_type_enum.name.lower()] = df_group
    return metadata_lib.AlphaGenomeOutputMetadata(**metadata)

output_metadata = {
    dna_model.Organism.HOMO_SAPIENS: build_output_metadata(TRACK_METADATA)
}
print("Output metadata built!")


Track metadata:
   output_type                                               name strand  \
0      RNA_SEQ  MM_BM061418.Aligned.sortedByCoord.out total RN...      .   
1      RNA_SEQ  MM_BM061472.Aligned.sortedByCoord.out total RN...      .   
2      RNA_SEQ  MM_BM63568R.Aligned.sortedByCoord.out total RN...      .   
3      RNA_SEQ  MM_BM64752R.Aligned.sortedByCoord.out total RN...      .   
4      RNA_SEQ  MM_BM65549R.Aligned.sortedByCoord.out total RN...      .   
5      RNA_SEQ  MM_BM65562R.Aligned.sortedByCoord.out total RN...      .   
6      RNA_SEQ  MM_BM65678R.Aligned.sortedByCoord.out total RN...      .   
7      RNA_SEQ  MM_BM65904R.Aligned.sortedByCoord.out total RN...      .   
8      RNA_SEQ  MM_BM66361R.Aligned.sortedByCoord.out total RN...      .   
9      RNA_SEQ  MM_BM67614R.Aligned.sortedByCoord.out total RN...      .   
10     RNA_SEQ  MM_BM68182R.Aligned.sortedByCoord.out total RN...      .   
11     RNA_SEQ  MM_BM68266R.Aligned.sortedByCoord.out total RN...      

In [7]:
from alphagenome_research.model.metadata import metadata as metadata_lib
print(dir(metadata_lib))

['AlphaGenomeOutputMetadata', 'Bool', 'Collection', 'Int32', 'Mapping', '_PADDING_TRACK_NAME', '_PATH_METADATA', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_create_output_strand_reindexing', 'create_track_masks', 'dataclasses', 'dna_client', 'dna_model', 'dna_model_service_pb2', 'dna_output', 'functools', 'load', 'np', 'ontology', 'pathlib', 'resources', 'text_format', 'track_data', 'typing']


In [8]:

from alphagenome_research.finetuning import finetune

# metadata is nested
from alphagenome_research.model.metadata import metadata as metadata_lib

# Check AlphaGenomeOutputMetadata
print([m for m in dir(metadata_lib) if not m.startswith('_')])

['AlphaGenomeOutputMetadata', 'Bool', 'Collection', 'Int32', 'Mapping', 'create_track_masks', 'dataclasses', 'dna_client', 'dna_model', 'dna_model_service_pb2', 'dna_output', 'functools', 'load', 'np', 'ontology', 'pathlib', 'resources', 'text_format', 'track_data', 'typing']


In [ ]:
# Load base model
SEQUENCE_LENGTH = int(2**17)
ORGANISM = dna_model.Organism.HOMO_SAPIENS

# Load base weights
checkpointer = ocp.StandardCheckpointer()
params_base, state_base = checkpointer.restore('/content/alphagenome-jax-all_folds-v1')
print("Base weights loaded!")

# Create fine-tuned model
default_settings_human = dna_model.default_organism_settings()[
    dna_model.Organism.HOMO_SAPIENS
]
settings_human_finetune = dataclasses.replace(
    default_settings_human,
    metadata=output_metadata[dna_model.Organism.HOMO_SAPIENS],
)
model = dna_model.create(
    '/content/checkpoint_mm/',
    organism_settings={
        dna_model.Organism.HOMO_SAPIENS: settings_human_finetune
    },
)
print("✅ Fine-tuned model loaded!")

In [11]:
# Extract finetuned model
import os
import zipfile

os.makedirs('/content/checkpoint_mm', exist_ok=True)
with zipfile.ZipFile('/content/checkpoint.zip', 'r') as z:
    z.extractall('/content/checkpoint_mm/')

print("Extracted!")
!ls /content/checkpoint_mm/

Extracted!
array_metadatas       d		      _METADATA        _sharding
_CHECKPOINT_METADATA  manifest.ocdbt  ocdbt.process_0


In [15]:
# Extract base model
import os

# Extract all_folds tar.gz
os.makedirs('/content/alphagenome_all_folds', exist_ok=True)
print("Extracting all_folds weights...")
!tar -xzf /content/alphagenome-jax-all_folds-v1.tar.gz \
    -C /content/alphagenome_all_folds/
print("Done!")
!ls /content/alphagenome_all_folds/

Extracting all_folds weights...
Done!
_CHECKPOINT_METADATA  d  manifest.ocdbt  _METADATA  ocdbt.process_0


In [18]:
!find /content/alphagenome_all_folds/ -name "_METADATA" 2>/dev/null



/content/alphagenome_all_folds/_METADATA
